# RQ3-05 — Generate 1,000 AI Essays via OpenRouter

**Trạng thái project:** các detector RQ3 đã được retrain và checkpoint đã được lưu.  
Notebook này chỉ tạo **1,000 AI essays mới** cho external benchmark. Sau khi generate xong, **không train/tune checkpoint thêm**.

## Một API key, 5 fixed model families

Tất cả model được gọi qua **OpenRouter** bằng `OPENROUTER_API_KEY`.

| Family | Fixed OpenRouter model slug |
|---|---|
| GPT | `openai/gpt-5.6-terra` |
| Claude | `anthropic/claude-sonnet-5.5` |
| Gemini | `google/gemini-3.8-flash` |
| Llama | `meta-llama/llama-3.3-70b-instruct` |
| DeepSeek | `deepseek/deepseek-v4-pro-0813` |

Không dùng `openrouter/auto` và không dùng alias `latest`.

### Thiết kế 1,000 final AI essays

\[
10\ prompts \times 5\ model\ families \times 4\ conditions \times 5\ replicates
= 1000
\]

Conditions:
1. `standard`
2. `natural`
3. `student_style`
4. `post_edited`

`post_edited` dùng chính bài `standard` làm parent, sau đó **một model family khác** chỉnh nhẹ.  
Do đó tổng số API calls vẫn là:

- 750 direct generation calls
- 250 post-edit calls
- **1,000 calls tổng cộng**

### Budget

Hard cap: **$20**.  
Notebook dùng safety reserve $2 và dừng khi recorded estimated spend đạt $18.

Pricing snapshot dùng để guard chi phí:
- GPT-5.6 Terra: $2 input / $12 output per 1M tokens
- Claude Sonnet 5.5: $2 / $10
- Gemini 3.8 Flash: $0.75 / $3.75
- Llama 3.3 70B Instruct: $0.10 / $0.32
- DeepSeek V4 Pro 0813: $0.5267 / $1.58

OpenRouter Standard có platform fee trên lúc mua credit; token inference được tính theo model/provider catalog. Giá có thể thay đổi, nên kiểm tra lại OpenRouter dashboard trước production.

> **Terminology:** in this notebook there is **1 API host/key (OpenRouter)** and **5 fixed generator models/families**. The experiment should report the five model families, while `api_host=OpenRouter` records how they were accessed.


## 0. Cài package

OpenRouter dùng API tương thích OpenAI nên chỉ cần `openai` SDK.

In [ ]:
%pip install -q -U openai python-dotenv pandas numpy tqdm requests


## 1. Imports và project paths

In [ ]:
import os
import re
import json
import time
import random
import requests
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

if Path("/content/AI-Generated-Text-Essay-Detection").exists():
    PROJECT_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")

RQ3_DIR = PROJECT_ROOT / "data" / "rq3"
RQ3_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_FILE = RQ3_DIR / "rq3_prompts.csv"
AI_OUTPUT_FILE = RQ3_DIR / "rq3_ai_1000.csv"
ERROR_LOG_FILE = RQ3_DIR / "rq3_generation_errors.jsonl"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RQ3_DIR      :", RQ3_DIR)
print("AI output    :", AI_OUTPUT_FILE)

## 2. Experimental config

Lần đầu giữ `RUN_MODE="dry_run"`.

- `dry_run`: không gọi API.
- `smoke`: đúng 5 calls, 1 Standard essay/model.
- `production`: resume và chạy tới đủ 1,000 final essays.

`ROUTING_SORT="price"` giúp giảm chi phí. OpenRouter vẫn có thể failover sang serving provider khác; notebook ghi lại `serving_provider` nếu response trả trường đó.

In [ ]:
RUN_MODE = "dry_run"  # "dry_run" | "smoke" | "production"

BUDGET_USD = 20.0
SAFETY_RESERVE_USD = 2.0
USABLE_BUDGET_USD = BUDGET_USD - SAFETY_RESERVE_USD

REPLICATES = 5
TARGET_MIN_WORDS = 400
TARGET_MAX_WORDS = 600
ABSOLUTE_MIN_WORDS = 250
MAX_OUTPUT_TOKENS = 1800

DIRECT_CONDITIONS = ["standard", "natural", "student_style"]
ALL_CONDITIONS = DIRECT_CONDITIONS + ["post_edited"]

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
ROUTING_SORT = "price"

MODELS = {
    "gpt": {
        "family": "GPT",
        "model": "openai/gpt-5.6-terra",
        "developer": "OpenAI",
        "input_per_m": 2.00,
        "output_per_m": 12.00,
    },
    "claude": {
        "family": "Claude",
        "model": "anthropic/claude-sonnet-5.5",
        "developer": "Anthropic",
        "input_per_m": 2.00,
        "output_per_m": 10.00,
    },
    "gemini": {
        "family": "Gemini",
        "model": "google/gemini-3.8-flash",
        "developer": "Google",
        "input_per_m": 0.75,
        "output_per_m": 3.75,
    },
    "llama": {
        "family": "Llama",
        "model": "meta-llama/llama-3.3-70b-instruct",
        "developer": "Meta",
        "input_per_m": 0.10,
        "output_per_m": 0.32,
    },
    "deepseek": {
        "family": "DeepSeek",
        "model": "deepseek/deepseek-v4-pro-0813",
        "developer": "DeepSeek",
        "input_per_m": 0.5267,
        "output_per_m": 1.58,
    },
}

# Standard parent from one family is lightly edited by another family.
EDITOR_MAP = {
    "gpt": "claude",
    "claude": "gemini",
    "gemini": "llama",
    "llama": "deepseek",
    "deepseek": "gpt",
}

assert set(EDITOR_MAP) == set(MODELS)
assert set(EDITOR_MAP.values()) == set(MODELS)

print("Model families:", list(MODELS))
print("Hard budget   : $", BUDGET_USD)
print("Usable budget : $", USABLE_BUDGET_USD)

## 3. Prompt file

Cần file:

`data/rq3/rq3_prompts.csv`

Schema:
- `prompt_id`
- `prompt_name`
- `assignment`
- `requires_source`
- `source_text`
- `verified`

Không chạy production nếu assignment/source chưa được xác minh.

In [ ]:
SELECTED_PROMPTS = [
    ("P01", "Phones and driving", False),
    ("P02", "Distance learning", False),
    ("P03", "Summer projects", False),
    ("P04", "Mandatory extracurricular activities", False),
    ("P05", "Cell phones at school", False),
    ("P06", "Grades for extracurricular activities", False),
    ("P07", "Seeking multiple opinions", False),
    ("P08", "Community service", False),
    ("P09", "Driverless cars", True),
    ("P10", "Does the electoral college work?", True),
]

if not PROMPTS_FILE.exists():
    template = pd.DataFrame([
        {
            "prompt_id": pid,
            "prompt_name": name,
            "assignment": "TODO: paste exact assignment",
            "requires_source": needs_source,
            "source_text": "TODO: paste required source material" if needs_source else "",
            "verified": False,
        }
        for pid, name, needs_source in SELECTED_PROMPTS
    ])
    template.to_csv(PROMPTS_FILE, index=False)
    print("Created template:", PROMPTS_FILE)

prompts_df = pd.read_csv(PROMPTS_FILE).fillna("")
display(prompts_df)

In [ ]:
def as_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {"1", "true", "yes", "y"}

prompts_df["verified"] = prompts_df["verified"].map(as_bool)
prompts_df["requires_source"] = prompts_df["requires_source"].map(as_bool)

required_cols = {
    "prompt_id", "prompt_name", "assignment",
    "requires_source", "source_text", "verified"
}
missing_cols = required_cols - set(prompts_df.columns)
if missing_cols:
    raise ValueError(f"rq3_prompts.csv thiếu cột: {missing_cols}")

if len(prompts_df) != 10:
    raise ValueError(f"Cần đúng 10 prompts, hiện có {len(prompts_df)}")

def prompt_audit(frame):
    problems = []
    for _, r in frame.iterrows():
        assignment = str(r["assignment"]).strip()
        source = str(r["source_text"]).strip()
        if not r["verified"]:
            problems.append((r["prompt_id"], "verified=False"))
        if not assignment or assignment.upper().startswith("TODO"):
            problems.append((r["prompt_id"], "assignment chưa điền"))
        if r["requires_source"] and (not source or source.upper().startswith("TODO")):
            problems.append((r["prompt_id"], "source_text chưa điền"))
    return problems

PROMPT_PROBLEMS = prompt_audit(prompts_df)

if PROMPT_PROBLEMS:
    print("Prompt audit chưa pass:")
    for x in PROMPT_PROBLEMS:
        print(" -", x)
else:
    print("✅ Prompt audit PASS")

if RUN_MODE in {"smoke", "production"} and PROMPT_PROBLEMS:
    raise RuntimeError(
        "Không gọi API khi prompt assignment/source chưa verified."
    )

## 4. OpenRouter API key

Chỉ cần:

`OPENROUTER_API_KEY`

Có thể đặt trong `.env` hoặc Colab Secrets. Notebook không in giá trị key.

In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=False)

if not os.getenv("OPENROUTER_API_KEY"):
    try:
        from google.colab import userdata
        value = userdata.get("OPENROUTER_API_KEY")
        if value:
            os.environ["OPENROUTER_API_KEY"] = value
    except Exception:
        pass

HAS_KEY = bool(os.getenv("OPENROUTER_API_KEY"))
print("OPENROUTER_API_KEY found:", HAS_KEY)

if RUN_MODE in {"smoke", "production"} and not HAS_KEY:
    raise RuntimeError("Thiếu OPENROUTER_API_KEY")

## 4B. Strict model verification — chạy trước Smoke/Production

Cell dưới đây bảo đảm:

1. Chính xác **5 model keys**: GPT, Claude, Gemini, Llama, DeepSeek.
2. Không dùng `openrouter/auto`, `latest`, wildcard hoặc model fallback list.
3. Tất cả fixed model slugs đang tồn tại trong OpenRouter catalog.
4. Production dừng ngay nếu một slug bị đổi/xóa.

Lưu ý: OpenRouter vẫn có thể chuyển **serving provider** (ví dụ provider A → provider B)
cho cùng một model để failover. Đây không phải đổi model family.

In [ ]:
EXPECTED_FIXED_MODELS = {
    "gpt": "openai/gpt-5.6-terra",
    "claude": "anthropic/claude-sonnet-5.5",
    "gemini": "google/gemini-3.8-flash",
    "llama": "meta-llama/llama-3.3-70b-instruct",
    "deepseek": "deepseek/deepseek-v4-pro-0813",
}

def verify_model_config():
    # Exact experiment definition.
    assert set(MODELS) == set(EXPECTED_FIXED_MODELS), (
        f"Model keys changed. Expected {set(EXPECTED_FIXED_MODELS)}, "
        f"found {set(MODELS)}"
    )

    configured = {k: v["model"] for k, v in MODELS.items()}
    assert configured == EXPECTED_FIXED_MODELS, (
        "Configured model slugs differ from the approved RQ3 model set.\n"
        f"Expected: {EXPECTED_FIXED_MODELS}\n"
        f"Found:    {configured}"
    )

    # Explicitly reject dynamic routing/model aliases.
    forbidden_tokens = ["openrouter/auto", "/auto", "latest", "*"]
    for key, slug in configured.items():
        low = slug.lower()
        assert not any(token in low for token in forbidden_tokens), (
            f"{key}: dynamic/alias model is not allowed: {slug}"
        )

    assert len(set(configured.values())) == 5, "The 5 model slugs must be unique."

    # Verify against current OpenRouter model catalog.
    headers = {}
    api_key = os.getenv("OPENROUTER_API_KEY")
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    response = requests.get(
        "https://openrouter.ai/api/v1/models",
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()

    payload = response.json()
    available_ids = {item["id"] for item in payload.get("data", [])}

    missing = [
        slug for slug in configured.values()
        if slug not in available_ids
    ]
    if missing:
        raise RuntimeError(
            "These approved model slugs are not currently present in the "
            f"OpenRouter catalog: {missing}"
        )

    print("✅ STRICT MODEL CONFIG PASS")
    for key, slug in configured.items():
        print(f"  {key:8s} -> {slug}")
    return configured

VERIFIED_MODELS = verify_model_config()

## 5. Cost projection

In [ ]:
EST_DIRECT_INPUT = 300
EST_EDIT_INPUT = 1200
EST_OUTPUT = 900

n_prompts = len(prompts_df)
direct_calls_per_model = n_prompts * len(DIRECT_CONDITIONS) * REPLICATES
edit_calls_per_model = n_prompts * REPLICATES
calls_per_model = direct_calls_per_model + edit_calls_per_model

projection_rows = []
for key, cfg in MODELS.items():
    input_tokens = (
        direct_calls_per_model * EST_DIRECT_INPUT
        + edit_calls_per_model * EST_EDIT_INPUT
    )
    output_tokens = calls_per_model * EST_OUTPUT
    est_cost = (
        input_tokens / 1_000_000 * cfg["input_per_m"]
        + output_tokens / 1_000_000 * cfg["output_per_m"]
    )
    projection_rows.append({
        "model_key": key,
        "family": cfg["family"],
        "model": cfg["model"],
        "direct_calls": direct_calls_per_model,
        "edit_calls": edit_calls_per_model,
        "total_calls": calls_per_model,
        "est_input_tokens": input_tokens,
        "est_output_tokens": output_tokens,
        "est_cost_usd": round(est_cost, 4),
    })

projection = pd.DataFrame(projection_rows)
display(projection)

projected_total = projection["est_cost_usd"].sum()
print("Projected inference total: $", round(projected_total, 4))
print("Usable budget             : $", USABLE_BUDGET_USD)

assert direct_calls_per_model == 150
assert edit_calls_per_model == 50
assert calls_per_model == 200
assert calls_per_model * len(MODELS) == 1000
assert projected_total < USABLE_BUDGET_USD

## 6. Prompt builders

In [ ]:
BASE_SYSTEM = (
    "You are producing an essay response for an academic robustness benchmark. "
    "Follow the assignment faithfully. Return only the essay text. "
    "Do not include analysis, task headings, or meta-commentary."
)

CONDITION_INSTRUCTIONS = {
    "standard": (
        "Write a clear and coherent argumentative essay. "
        "Use a conventional introduction, body, and conclusion where appropriate."
    ),
    "natural": (
        "Write naturally with varied sentence lengths and vocabulary. "
        "Avoid repetitive transitions and overly formulaic paragraph structure."
    ),
    "student_style": (
        "Write in a plausible secondary-school student voice. "
        "Keep the reasoning understandable and specific without making the prose "
        "artificially polished or professional."
    ),
}

def direct_prompt(row, condition):
    if condition not in DIRECT_CONDITIONS:
        raise ValueError(condition)

    parts = [f"Assignment:\n{row['assignment']}"]
    source = str(row.get("source_text", "")).strip()
    if source:
        parts.append(f"Source material:\n{source}")

    parts.append(f"Writing condition:\n{CONDITION_INSTRUCTIONS[condition]}")
    parts.append(
        f"Length: approximately {TARGET_MIN_WORDS}-{TARGET_MAX_WORDS} words. "
        "Return only the final essay."
    )
    return "\n\n".join(parts)

def edit_prompt(parent_text, row):
    source = str(row.get("source_text", "")).strip()
    source_block = f"\n\nSource material:\n{source}" if source else ""

    return f'''
Assignment:
{row["assignment"]}
{source_block}

Existing essay:
--- BEGIN ESSAY ---
{parent_text}
--- END ESSAY ---

Lightly revise the essay while preserving its core position, evidence, and meaning.
Make the prose read more naturally by varying sentence structure, transitions, and wording.
Keep a plausible student voice. Do not mention editing, AI, language models, or detection.
Do not add meta-commentary.

Keep the final essay approximately {TARGET_MIN_WORDS}-{TARGET_MAX_WORDS} words.
Return only the revised essay.
'''.strip()

## 7. OpenRouter client

Một client duy nhất cho cả 5 model slugs.

Notebook:
- không dùng Auto Router model;
- dùng fixed model slug;
- yêu cầu OpenRouter ưu tiên route rẻ (`provider.sort="price"`);
- ghi `response_id`, resolved model và serving provider nếu OpenRouter trả về.

In [ ]:
_client = None

def get_client():
    global _client
    if _client is None:
        key = os.getenv("OPENROUTER_API_KEY")
        if not key:
            raise RuntimeError("Missing OPENROUTER_API_KEY")
        _client = OpenAI(
            api_key=key,
            base_url=OPENROUTER_BASE_URL,
            max_retries=0,
            timeout=120.0,
        )
    return _client

def token_estimated_cost(model_key, input_tokens, output_tokens):
    cfg = MODELS[model_key]
    return (
        int(input_tokens or 0) / 1_000_000 * cfg["input_per_m"]
        + int(output_tokens or 0) / 1_000_000 * cfg["output_per_m"]
    )

def call_openrouter(model_key, user_prompt):
    cfg = MODELS[model_key]
    client = get_client()

    response = client.chat.completions.create(
        model=cfg["model"],
        messages=[
            {"role": "system", "content": BASE_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=MAX_OUTPUT_TOKENS,
        extra_body={
            "provider": {
                "sort": ROUTING_SORT,
                "allow_fallbacks": True,
            },
            "usage": {"include": True},
        },
    )

    raw = response.model_dump()
    text = (response.choices[0].message.content or "").strip()

    requested_model = cfg["model"]
    resolved_model = str(raw.get("model", "")).strip()

    # Research guard: do not accept a response attributed to another model.
    if not resolved_model:
        raise RuntimeError(
            f"{model_key}: OpenRouter response did not report a resolved model."
        )
    if resolved_model != requested_model:
        raise RuntimeError(
            f"{model_key}: MODEL MISMATCH. "
            f"requested={requested_model!r}, resolved={resolved_model!r}"
        )

    if not text:
        raise RuntimeError(f"{model_key} returned empty text")

    usage = response.usage
    input_tokens = int(getattr(usage, "prompt_tokens", 0) or 0)
    output_tokens = int(getattr(usage, "completion_tokens", 0) or 0)

    reasoning_tokens = 0
    details = getattr(usage, "completion_tokens_details", None)
    if details is not None:
        reasoning_tokens = int(getattr(details, "reasoning_tokens", 0) or 0)

    cost = token_estimated_cost(model_key, input_tokens, output_tokens)

    return {
        "text": text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "estimated_cost_usd": cost,
        "response_id": str(raw.get("id", "")),
        "resolved_model": resolved_model,
        "serving_provider": str(raw.get("provider", "")),
    }

## 8. Retry policy

In [ ]:
MAX_RETRIES = 4
BASE_BACKOFF_SECONDS = 3

PERMANENT_PATTERNS = [
    "insufficient credits",
    "insufficient_quota",
    "payment required",
    "billing",
    "credit",
    "account deactivated",
]

TRANSIENT_PATTERNS = [
    "rate limit",
    "429",
    "timeout",
    "timed out",
    "connection",
    "internal server",
    "502",
    "503",
    "504",
    "temporarily unavailable",
]

class ModelBlocked(RuntimeError):
    pass

def classify_error(exc):
    msg = str(exc).lower()
    if any(p in msg for p in PERMANENT_PATTERNS):
        return "permanent"
    if any(p in msg for p in TRANSIENT_PATTERNS):
        return "transient"
    return "unknown"

def append_error(payload):
    row = dict(payload)
    row["timestamp_utc"] = datetime.now(timezone.utc).isoformat()
    with open(ERROR_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def call_with_retry(model_key, prompt, context=None):
    context = context or {}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call_openrouter(model_key, prompt)
        except Exception as exc:
            kind = classify_error(exc)
            append_error({
                **context,
                "model_key": model_key,
                "attempt": attempt,
                "error_kind": kind,
                "error": str(exc)[:2000],
            })

            if kind == "permanent":
                raise ModelBlocked(
                    f"{model_key} blocked by credits/billing: {exc}"
                ) from exc

            if attempt == MAX_RETRIES:
                raise

            sleep_s = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1))
            print(f"[{model_key}] attempt {attempt} failed ({kind}); retry in {sleep_s}s")
            time.sleep(sleep_s)

def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))

def deterministic_id(prefix, *parts):
    raw = "||".join(map(str, parts))
    return f"{prefix}_{hashlib.sha256(raw.encode()).hexdigest()[:14]}"

## 9. Resume + schema

In [ ]:
RESULT_COLUMNS = [
    "essay_id",
    "label",
    "prompt_id",
    "prompt_name",
    "condition",
    "replicate",

    "generator_key",
    "generator_family",
    "generator_developer",
    "generator_model",

    "final_key",
    "final_family",
    "final_developer",
    "final_model",

    "editor_key",
    "editor_model",
    "parent_essay_id",

    "api_host",
    "serving_provider",
    "resolved_model",
    "response_id",

    "text",
    "word_count",
    "input_tokens",
    "output_tokens",
    "reasoning_tokens",
    "estimated_cost_usd",
    "quality_status",
    "created_at_utc",
]

def load_results():
    if not AI_OUTPUT_FILE.exists():
        return pd.DataFrame(columns=RESULT_COLUMNS)

    frame = pd.read_csv(AI_OUTPUT_FILE).fillna("")

    # Prevent accidentally resuming an older 5-direct-API dataset.
    if "api_host" not in frame.columns:
        raise RuntimeError(
            f"{AI_OUTPUT_FILE} tồn tại nhưng không phải OpenRouter schema. "
            "Đổi tên/move file cũ trước khi chạy notebook này."
        )

    hosts = set(frame["api_host"].astype(str).str.lower().unique())
    if hosts - {"openrouter"}:
        raise RuntimeError(
            f"Không mix API hosts trong cùng benchmark. Found: {hosts}"
        )

    for c in RESULT_COLUMNS:
        if c not in frame.columns:
            frame[c] = ""
    return frame[RESULT_COLUMNS].copy()

def save_results(frame):
    tmp = AI_OUTPUT_FILE.with_suffix(".tmp.csv")
    frame.to_csv(tmp, index=False)
    tmp.replace(AI_OUTPUT_FILE)

def result_key(generator_key, prompt_id, condition, replicate):
    return f"{generator_key}|{prompt_id}|{condition}|{replicate}"

def completed_keys(frame):
    if frame.empty:
        return set()
    return {
        result_key(
            r.generator_key,
            r.prompt_id,
            r.condition,
            int(r.replicate),
        )
        for r in frame.itertuples()
    }

def current_spend(frame):
    if frame.empty:
        return 0.0
    return float(
        pd.to_numeric(frame["estimated_cost_usd"], errors="coerce")
        .fillna(0).sum()
    )

results_df = load_results()
print("Existing OpenRouter rows:", len(results_df))
print("Recorded estimated spend: $", round(current_spend(results_df), 4))

## 10. Job plan

In [ ]:
def build_direct_jobs():
    jobs = []
    for model_key in MODELS:
        for _, p in prompts_df.iterrows():
            for condition in DIRECT_CONDITIONS:
                for rep in range(1, REPLICATES + 1):
                    jobs.append({
                        "generator_key": model_key,
                        "prompt_id": p["prompt_id"],
                        "condition": condition,
                        "replicate": rep,
                    })
    return jobs

def build_post_edit_jobs():
    jobs = []
    for base_key in MODELS:
        for _, p in prompts_df.iterrows():
            for rep in range(1, REPLICATES + 1):
                jobs.append({
                    "generator_key": base_key,
                    "prompt_id": p["prompt_id"],
                    "condition": "post_edited",
                    "replicate": rep,
                    "editor_key": EDITOR_MAP[base_key],
                })
    return jobs

direct_jobs = build_direct_jobs()
post_jobs = build_post_edit_jobs()

print("Direct jobs :", len(direct_jobs))
print("Post-edit   :", len(post_jobs))
print("TOTAL       :", len(direct_jobs) + len(post_jobs))

assert len(direct_jobs) == 750
assert len(post_jobs) == 250
assert len(direct_jobs) + len(post_jobs) == 1000

## 11. Generation engine

In [ ]:
prompt_lookup = prompts_df.set_index("prompt_id").to_dict("index")

def ensure_budget(frame):
    spent = current_spend(frame)
    if spent >= USABLE_BUDGET_USD:
        raise RuntimeError(
            f"Budget guard triggered: ${spent:.4f} >= ${USABLE_BUDGET_USD:.2f}"
        )
    return spent

def append_result(frame, row):
    frame = pd.concat([frame, pd.DataFrame([row])], ignore_index=True)
    save_results(frame)
    return frame

def make_direct_row(job, api_result):
    key = job["generator_key"]
    cfg = MODELS[key]
    p = prompt_lookup[job["prompt_id"]]
    wc = word_count(api_result["text"])

    return {
        "essay_id": deterministic_id(
            "AI", key, job["prompt_id"], job["condition"], job["replicate"]
        ),
        "label": 1,
        "prompt_id": job["prompt_id"],
        "prompt_name": p["prompt_name"],
        "condition": job["condition"],
        "replicate": job["replicate"],

        "generator_key": key,
        "generator_family": cfg["family"],
        "generator_developer": cfg["developer"],
        "generator_model": cfg["model"],

        "final_key": key,
        "final_family": cfg["family"],
        "final_developer": cfg["developer"],
        "final_model": cfg["model"],

        "editor_key": "",
        "editor_model": "",
        "parent_essay_id": "",

        "api_host": "OpenRouter",
        "serving_provider": api_result["serving_provider"],
        "resolved_model": api_result["resolved_model"],
        "response_id": api_result["response_id"],

        "text": api_result["text"],
        "word_count": wc,
        "input_tokens": api_result["input_tokens"],
        "output_tokens": api_result["output_tokens"],
        "reasoning_tokens": api_result["reasoning_tokens"],
        "estimated_cost_usd": api_result["estimated_cost_usd"],
        "quality_status": "ok" if wc >= ABSOLUTE_MIN_WORDS else "short",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

def find_standard_parent(frame, base_key, prompt_id, replicate):
    hit = frame[
        (frame["generator_key"] == base_key)
        & (frame["prompt_id"] == prompt_id)
        & (frame["condition"] == "standard")
        & (pd.to_numeric(frame["replicate"], errors="coerce") == replicate)
    ]
    return None if hit.empty else hit.iloc[-1]

def make_post_edit_row(job, parent, api_result):
    base_key = job["generator_key"]
    editor_key = job["editor_key"]
    base_cfg = MODELS[base_key]
    editor_cfg = MODELS[editor_key]
    p = prompt_lookup[job["prompt_id"]]
    wc = word_count(api_result["text"])

    return {
        "essay_id": deterministic_id(
            "AI", base_key, job["prompt_id"], "post_edited", job["replicate"]
        ),
        "label": 1,
        "prompt_id": job["prompt_id"],
        "prompt_name": p["prompt_name"],
        "condition": "post_edited",
        "replicate": job["replicate"],

        "generator_key": base_key,
        "generator_family": base_cfg["family"],
        "generator_developer": base_cfg["developer"],
        "generator_model": base_cfg["model"],

        "final_key": editor_key,
        "final_family": editor_cfg["family"],
        "final_developer": editor_cfg["developer"],
        "final_model": editor_cfg["model"],

        "editor_key": editor_key,
        "editor_model": editor_cfg["model"],
        "parent_essay_id": parent["essay_id"],

        "api_host": "OpenRouter",
        "serving_provider": api_result["serving_provider"],
        "resolved_model": api_result["resolved_model"],
        "response_id": api_result["response_id"],

        "text": api_result["text"],
        "word_count": wc,
        "input_tokens": api_result["input_tokens"],
        "output_tokens": api_result["output_tokens"],
        "reasoning_tokens": api_result["reasoning_tokens"],
        "estimated_cost_usd": api_result["estimated_cost_usd"],
        "quality_status": "ok" if wc >= ABSOLUTE_MIN_WORDS else "short",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

def run_generation(run_mode=RUN_MODE):
    global results_df

    if run_mode == "dry_run":
        print("DRY RUN — no API calls.")
        return results_df

    if PROMPT_PROBLEMS:
        raise RuntimeError("Prompt audit chưa PASS.")
    if not os.getenv("OPENROUTER_API_KEY"):
        raise RuntimeError("Missing OPENROUTER_API_KEY")

    blocked_models = set()
    done = completed_keys(results_df)

    jobs = direct_jobs
    if run_mode == "smoke":
        first_prompt = prompts_df.iloc[0]["prompt_id"]
        jobs = [
            {
                "generator_key": key,
                "prompt_id": first_prompt,
                "condition": "standard",
                "replicate": 1,
            }
            for key in MODELS
        ]

    print(f"Direct stage: {len(jobs)} planned jobs")
    for job in tqdm(jobs):
        key = result_key(
            job["generator_key"],
            job["prompt_id"],
            job["condition"],
            job["replicate"],
        )
        if key in done:
            continue

        model_key = job["generator_key"]
        if model_key in blocked_models:
            continue

        ensure_budget(results_df)
        p = prompt_lookup[job["prompt_id"]]
        prompt = direct_prompt(p, job["condition"])

        try:
            api_result = call_with_retry(
                model_key,
                prompt,
                context={"job_key": key, "stage": "direct"},
            )
            results_df = append_result(
                results_df, make_direct_row(job, api_result)
            )
            done.add(key)

        except ModelBlocked as exc:
            blocked_models.add(model_key)
            print("BLOCKED:", exc)

        except Exception as exc:
            print("FAILED:", key, type(exc).__name__, str(exc)[:300])

    if run_mode == "smoke":
        print("✅ Smoke mode complete. Production/post-edit not started.")
        return results_df

    print(f"Post-edit stage: {len(post_jobs)} planned jobs")
    for job in tqdm(post_jobs):
        key = result_key(
            job["generator_key"],
            job["prompt_id"],
            "post_edited",
            job["replicate"],
        )
        if key in done:
            continue

        editor_key = job["editor_key"]
        if editor_key in blocked_models:
            continue

        parent = find_standard_parent(
            results_df,
            job["generator_key"],
            job["prompt_id"],
            job["replicate"],
        )
        if parent is None:
            continue

        ensure_budget(results_df)
        p = prompt_lookup[job["prompt_id"]]
        prompt = edit_prompt(parent["text"], p)

        try:
            api_result = call_with_retry(
                editor_key,
                prompt,
                context={
                    "job_key": key,
                    "stage": "post_edit",
                    "parent_essay_id": parent["essay_id"],
                },
            )
            results_df = append_result(
                results_df, make_post_edit_row(job, parent, api_result)
            )
            done.add(key)

        except ModelBlocked as exc:
            blocked_models.add(editor_key)
            print("BLOCKED:", exc)

        except Exception as exc:
            print("FAILED:", key, type(exc).__name__, str(exc)[:300])

    return results_df

results_df = run_generation()

## 12. Progress dashboard

In [ ]:
results_df = load_results()

if results_df.empty:
    print("Chưa có OpenRouter output.")
else:
    display(
        results_df.groupby(["generator_family", "condition"])
        .size().unstack(fill_value=0)
    )

    display(
        results_df.assign(
            estimated_cost_usd=pd.to_numeric(
                results_df["estimated_cost_usd"], errors="coerce"
            ).fillna(0)
        )
        .groupby("final_family")["estimated_cost_usd"]
        .sum()
        .sort_values(ascending=False)
        .to_frame("estimated_cost_usd")
    )

    print("Rows       :", len(results_df), "/ 1000")
    print("Est. spend : $", round(current_spend(results_df), 4))
    print("Short rows :", int((results_df["quality_status"] == "short").sum()))

## 13. Final validation

In [ ]:
results_df = load_results()

if len(results_df) == 1000:
    keys = (
        results_df["generator_key"].astype(str) + "|"
        + results_df["prompt_id"].astype(str) + "|"
        + results_df["condition"].astype(str) + "|"
        + results_df["replicate"].astype(str)
    )

    assert keys.nunique() == 1000
    assert (results_df.groupby("generator_family").size() == 200).all()
    assert (
        results_df.groupby(["generator_family", "condition"]).size() == 50
    ).all()
    assert (results_df.groupby("prompt_id").size() == 100).all()
    assert set(results_df["api_host"]) == {"OpenRouter"}

    # Every saved row must report exactly one of the 5 approved fixed models.
    approved_slugs = set(EXPECTED_FIXED_MODELS.values())
    observed_final_models = set(results_df["resolved_model"].astype(str))
    assert observed_final_models.issubset(approved_slugs), (
        f"Unexpected resolved models found: {observed_final_models - approved_slugs}"
    )

    expected_family_counts = {
        "GPT": 200,
        "Claude": 200,
        "Gemini": 200,
        "Llama": 200,
        "DeepSeek": 200,
    }
    actual_family_counts = results_df["generator_family"].value_counts().to_dict()
    assert actual_family_counts == expected_family_counts, (
        f"Family counts mismatch: {actual_family_counts}"
    )

    post = results_df[results_df["condition"] == "post_edited"]
    assert len(post) == 250
    assert (post["parent_essay_id"].astype(str).str.len() > 0).all()

    print("✅ COMPLETE: 1,000 OpenRouter AI essays")
    print("Estimated inference spend: $", round(current_spend(results_df), 4))
    print("Saved:", AI_OUTPUT_FILE)
else:
    print(f"Progress: {len(results_df)}/1000. Re-run production to resume.")

## 14. Sau khi đủ 1,000 AI

Không train checkpoint thêm.

Chuyển sang:

`13_RQ3_06_merge_and_evaluate_external.ipynb`

Notebook 13 sẽ ghép:
- `rq3_human_holdout_1000.csv`
- `rq3_ai_1000.csv`

thành external benchmark 2,000 essays và chạy inference bằng các checkpoint RQ3 đã retrain.